# 03 Well inflow performance (IPR): hydrogen vs methane at Rough

**Purpose.** Characterise how gas flows between the Rough reservoir and its 29 wells, for hydrogen and methane, using `h2_bottleneck.well`.

This notebook covers the **inflow (sandface) part of the well model only**: reservoir ↔ bottomhole. It does **not** yet include the tubing (bottomhole ↔ wellhead) or any surface equipment, so the field rates here are **upper bounds** on what the wells could deliver if any bottomhole pressure inside the operating envelope could be reached.

**Contents**
1. Pseudopressure m(p) for both gases
2. Non-Darcy coefficient D rescaled for hydrogen
3. Single-well IPR curves (withdrawal and injection)
4. How much of the pressure drop is turbulence (non-Darcy)?
5. Field sandface capacity against reservoir pressure, within the historic BHP envelope
6. Well-by-well ranking
7. Findings

**Equation** (see `well.py` docstring): m(p_res) − m(p_wf) = C·q·(L + s + D·|q|), with q > 0 withdrawal, q < 0 injection.

**Key assumptions:** A1 pure gases; A2/A9 single closed tank; A4 BHP limits = historic natural-gas envelope; A6 D rescaled for H₂; A11 field-average k and h; A12 pseudosteady radial inflow, r_e = 1500 m; A13 methane as the reference gas for Table 1 D-factors.

In [ ]:
%load_ext autoreload
%autoreload 2
import subprocess, hashlib, datetime, pathlib
root = pathlib.Path.cwd().parent
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=root).stdout.strip()
dirty = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True, cwd=root).stdout.strip() != ""
inp = hashlib.sha256((root / "data" / "inputs.csv").read_bytes()).hexdigest()[:10]
wls = hashlib.sha256((root / "data" / "wells.csv").read_bytes()).hexdigest()[:10]
print(f"Run {datetime.datetime.now():%Y-%m-%d %H:%M} | commit {commit}{' (UNCOMMITTED CHANGES)' if dirty else ''} | inputs.csv {inp} | wells.csv {wls}")

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from h2_bottleneck import inputs, props
from h2_bottleneck import well as wl

FIG = root / "results" / "figures"
TAB = root / "results" / "tables"
GASES = ("Hydrogen", "Methane")
COL = {"Hydrogen": "tab:blue", "Methane": "tab:orange"}
DAY = 86400.0                       # s per day
MSM3 = 1e6                          # Sm3 per million Sm3

x = inputs.load()
f = wl.Formation.from_inputs(x)
wdf = inputs.load_wells()
wells = [wl.Well.from_row(r) for _, r in wdf.iterrows()]
HHV = {"Hydrogen": x["hhv_hydrogen"], "Methane": x["hhv_methane"]}        # J/kg

t0 = time.time()
m = {g: wl.Pseudopressure(f.T, g) for g in GASES}                        # built once (CoolProp is slow)
print(f"Pseudopressure tables built in {time.time() - t0:.1f} s")
print(f"T = {f.T - 273.15:.1f} °C, k = {f.k / 9.869233e-16:.0f} mD, h = {f.h:.1f} m, r_e = {f.re:.0f} m")
print(f"C = {wl.darcy_constant(f):.3e} Pa/s per Sm3/s;  {len(wells)} wells loaded")

## 1. Pseudopressure

m(p) = 2∫ p/(μZ) dp combines pressure, viscosity and compressibility into one quantity, so a single inflow equation holds from low to high pressure. For an ideal gas with constant viscosity, m = p²/μ.

The **ratio m_H₂/m_CH₄** shows how much more "driving force" hydrogen gets from the same pressures, mainly because its viscosity is lower.

In [ ]:
p = np.linspace(10e5, 300e5, 300)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for g in GASES:
    ax[0].plot(p / 1e5, m[g](p) / 1e18, label=g, color=COL[g])
ax[0].set_xlabel("Pressure (bar)"); ax[0].set_ylabel("m(p) (10$^{18}$ Pa/s)"); ax[0].legend()
ax[0].set_title("Real-gas pseudopressure")
ax[1].plot(p / 1e5, m["Hydrogen"](p) / m["Methane"](p), color="k")
ax[1].set_xlabel("Pressure (bar)"); ax[1].set_ylabel("m$_{H2}$ / m$_{CH4}$")
ax[1].set_title("Hydrogen-to-methane ratio")
fig.tight_layout(); fig.savefig(FIG / "03_pseudopressure.png", dpi=300)

# Check: ideal-gas limit at low pressure (independent of the integration)
p1, p2 = 2e5, 5e5
mu = props.mu(3.5e5, f.T, "Hydrogen")
print(f"Ideal-gas check (H2, 2-5 bar): numerical {m['Hydrogen'](p2) - m['Hydrogen'](p1):.4e}, "
      f"analytical {(p2**2 - p1**2) / mu:.4e}")
for pb in (150, 220, 275):
    print(f"m_H2/m_CH4 at {pb} bar: {float(m['Hydrogen'](pb * 1e5) / m['Methane'](pb * 1e5)):.2f}")

## 2. Non-Darcy coefficient rescaled for hydrogen

Heinemann et al. (2025) Table 1 gives D measured with natural gas. D ∝ M/μ (Scafidi et al., 2023, eq. 5), so for the same well

D_H₂ = D_ref × (M_H₂/M_CH₄) × (μ_CH₄/μ_H₂).

The scaling factor is the same for every well at a given pressure; it varies only weakly with pressure.

In [ ]:
w0 = wells[0]
pp = np.linspace(100e5, 300e5, 41)
ratio = [wl.non_darcy_D(w0, pi, f.T, "Hydrogen") / w0.D_ref for pi in pp]
fig, ax = plt.subplots(figsize=(5.5, 3.5))
ax.plot(pp / 1e5, ratio, color="k")
ax.set_xlabel("Bottomhole pressure (bar)"); ax.set_ylabel("D$_{H2}$ / D$_{ref}$")
ax.set_title("Non-Darcy coefficient scaling for hydrogen")
fig.tight_layout(); fig.savefig(FIG / "03_D_scaling.png", dpi=300)
print(f"D_H2/D_ref: {min(ratio):.3f} - {max(ratio):.3f} over 100-300 bar")

D_table = pd.DataFrame({
    "well": [w.name for w in wells],
    "D_ref (s/Sm3)": [w.D_ref for w in wells],
    "D_H2 at 220 bar (s/Sm3)": [wl.non_darcy_D(w, 220e5, f.T, "Hydrogen") for w in wells],
})
D_table.describe().loc[["min", "50%", "max"]]

## 3. Single-well IPR curves

The **inflow performance relationship** is the rate a well delivers (or accepts) for each bottomhole pressure, at a fixed reservoir pressure. Positive rates are withdrawal (p_wf < p_res); negative rates are injection (p_wf > p_res).

Shown for well **IA02** (highest skin, s = 7), **IB04** (widest BHP envelope) and **IA04** (low skin, large wellbore), at p_res = 220 bar, between each well's historic BHP limits.

In [ ]:
p_res = 220e5
sel = {w.name: w for w in wells if w.name in ("IA02", "IB04", "IA04")}
fig, axes = plt.subplots(1, 3, figsize=(14, 3.9), sharey=True)
ipr_rows = []
for ax, (name, w) in zip(axes, sel.items()):
    pwf = np.linspace(w.p_bhp_min, w.p_bhp_max, 80)
    for g in GASES:
        q = np.array([wl.rate_from_pwf(pi, p_res, w, f, g, m[g]) for pi in pwf])
        ax.plot(q * DAY / MSM3, pwf / 1e5, color=COL[g], label=g)
        ipr_rows.append(dict(well=name, gas=g,
                             q_withdraw_at_bhp_min=q[0] * DAY / MSM3, q_inject_at_bhp_max=q[-1] * DAY / MSM3))
    ax.axhline(p_res / 1e5, color="grey", ls=":", lw=1)
    ax.axvline(0, color="grey", lw=0.8)
    ax.set_title(f"{name}  (s = {w.skin:g}, d = {2 * w.rw * 1000:.0f} mm)")
    ax.set_xlabel("Rate (million Sm$^3$/d)  ← injection | withdrawal →")
axes[0].set_ylabel("Bottomhole pressure (bar)"); axes[0].legend()
fig.suptitle("Inflow performance at p$_{res}$ = 220 bar, within historic BHP limits")
fig.tight_layout(); fig.savefig(FIG / "03_ipr_curves.png", dpi=300)
pd.DataFrame(ipr_rows).round(2)

**Reading the plot.** The curve crosses p_res at zero rate. Below it the well produces; above it the well accepts gas. Curvature comes from the non-Darcy term (∝ q²) and from μ and Z varying with pressure. A flatter curve means a more productive well.

## 4. How much of the pressure drop is turbulence?

The inflow resistance has a Darcy part, C(L + s), and a non-Darcy part, C·D·|q|. Their ratio shows whether turbulence matters at realistic rates, and how the reduced hydrogen D changes that.

In [ ]:
w = sel["IA02"]
q_d = np.linspace(0.1, 3.0, 60) * MSM3 / DAY          # 0.1-3 million Sm3/d per well
L = wl.geometry_term(f, w)
fig, ax = plt.subplots(figsize=(5.8, 3.6))
for g in GASES:
    D = np.array([wl.non_darcy_D(w, p_res, f.T, g) for _ in q_d])
    share = D * q_d / (L + w.skin + D * q_d)
    ax.plot(q_d * DAY / MSM3, 100 * share, color=COL[g], label=g)
ax.set_xlabel("Well rate (million Sm$^3$/d)"); ax.set_ylabel("Non-Darcy share of resistance (%)")
ax.set_title(f"{w.name}: turbulence share of inflow resistance"); ax.legend()
fig.tight_layout(); fig.savefig(FIG / "03_non_darcy_share.png", dpi=300)

## 5. Field sandface capacity against reservoir pressure

For each reservoir pressure, every well is set to its **historic BHP limit**:
- **withdrawal:** p_wf = BHP lower limit (only wells with p_res above it flow);
- **injection:** p_wf = BHP upper limit (only wells with p_res below it accept gas).

Summing the 29 wells gives the **maximum field rate the reservoir and wells could take or give, if the surface could hold every well at its limit**. Converted to energy flow (GWh/d, HHV) for a like-for-like comparison of the gases.

**Comparison values:**
- Historic methane: maximum withdrawal ≈ 44.8 million Sm³/d; average injection ≈ 15 million Sm³/d (Centrica, 2006).
- Notebook 02: hydrogen average rates needed to cycle 180 → 275 bar in Heinemann's schedule ≈ 21.1 million Sm³/d injection (120 d) and 28.1 million Sm³/d withdrawal (90 d).

In [ ]:
p_grid = np.linspace(150e5, 290e5, 29)
rows = []
for pr in p_grid:
    for g in GASES:
        q_wd = sum(wl.rate_from_pwf(w.p_bhp_min, pr, w, f, g, m[g]) for w in wells if pr > w.p_bhp_min)
        q_in = sum(-wl.rate_from_pwf(w.p_bhp_max, pr, w, f, g, m[g]) for w in wells if pr < w.p_bhp_max)
        e = props.rho_sc(g) * HHV[g] * DAY / 3.6e12                      # GWh per Sm3/s per day
        rows.append(dict(p_res_bar=pr / 1e5, gas=g,
                         withdrawal_MSm3d=q_wd * DAY / MSM3, injection_MSm3d=q_in * DAY / MSM3,
                         withdrawal_GWhd=q_wd * e, injection_GWhd=q_in * e))
cap = pd.DataFrame(rows)
cap.to_csv(TAB / "03_field_sandface_capacity.csv", index=False)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for g in GASES:
    c = cap[cap.gas == g]
    ax[0].plot(c.p_res_bar, c.withdrawal_MSm3d, color=COL[g], label=f"{g} withdrawal")
    ax[0].plot(c.p_res_bar, c.injection_MSm3d, color=COL[g], ls="--", label=f"{g} injection")
    ax[1].plot(c.p_res_bar, c.withdrawal_GWhd, color=COL[g], label=f"{g} withdrawal")
    ax[1].plot(c.p_res_bar, c.injection_GWhd, color=COL[g], ls="--", label=f"{g} injection")
ax[0].axhline(44.8, color="grey", ls=":", lw=1); ax[0].text(152, 46, "historic CH$_4$ max withdrawal", fontsize=8)
ax[0].axhline(21.1, color="tab:blue", ls=":", lw=1); ax[0].text(152, 22.5, "H$_2$ average injection needed (nb 02)", fontsize=8)
ax[0].set_ylabel("Field rate (million Sm$^3$/d)"); ax[1].set_ylabel("Field energy flow (GWh/d, HHV)")
for a in ax:
    a.set_xlabel("Average reservoir pressure (bar)"); a.axvspan(180, 275, color="0.92", zorder=0)
ax[0].legend(fontsize=8); ax[0].set_yscale("log"); ax[1].set_yscale("log")
fig.suptitle("Sandface capacity at historic BHP limits (no tubing or surface constraints); shaded = Heinemann H$_2$ cycle")
fig.tight_layout(); fig.savefig(FIG / "03_field_sandface_capacity.png", dpi=300)
cap[cap.p_res_bar.isin([180.0, 220.0, 275.0])].round(1)

**Points to check in the output:**
- How the injection capacity collapses as p_res approaches the BHP upper limits (282–306 bar), and withdrawal as it approaches the lower limits (98–185 bar). The usable window is set by the BHP envelope (A4).
- Whether the sandface capacity in volume terms is far above the historic methane rates. If it is, the wells' inflow is **not** what limited Rough historically, so tubing and surface facilities did.
- The energy comparison: hydrogen flows more volume but carries about 30 % of the energy per Sm³.

## 6. Well-by-well ranking at 220 bar

Which wells contribute most? Rates at each well's own BHP limit, at p_res = 220 bar.

In [ ]:
rank = []
for w in wells:
    r = {"well": w.name, "skin": w.skin, "d (mm)": round(2 * w.rw * 1000)}
    for g in GASES:
        r[f"{g} wd (MSm3/d)"] = wl.rate_from_pwf(w.p_bhp_min, 220e5, w, f, g, m[g]) * DAY / MSM3
        r[f"{g} inj (MSm3/d)"] = -wl.rate_from_pwf(w.p_bhp_max, 220e5, w, f, g, m[g]) * DAY / MSM3
    rank.append(r)
rank = pd.DataFrame(rank).sort_values("Hydrogen inj (MSm3/d)", ascending=False)
rank.to_csv(TAB / "03_well_ranking_220bar.csv", index=False)

fig, ax = plt.subplots(figsize=(12, 3.8))
xpos = np.arange(len(rank))
ax.bar(xpos - 0.2, rank["Hydrogen inj (MSm3/d)"], 0.4, color=COL["Hydrogen"], label="Hydrogen")
ax.bar(xpos + 0.2, rank["Methane inj (MSm3/d)"], 0.4, color=COL["Methane"], label="Methane")
ax.set_xticks(xpos, rank["well"], rotation=90); ax.set_ylabel("Injection rate (million Sm$^3$/d)")
ax.set_title("Per-well sandface injection at BHP upper limit, p$_{res}$ = 220 bar"); ax.legend()
fig.tight_layout(); fig.savefig(FIG / "03_well_ranking.png", dpi=300)
print("H2/CH4 volumetric injection ratio, all wells:",
      round(rank["Hydrogen inj (MSm3/d)"].sum() / rank["Methane inj (MSm3/d)"].sum(), 2))
rank.round(2).head(10)

## 7. Findings

_Fill in from the outputs above (use the exact values). Prompts:_

1. **Pseudopressure:** m_H₂/m_CH₄ = … at 220 bar, so for the same pressures hydrogen flows about … times the methane volume.
2. **Non-Darcy:** D_H₂ ≈ … × D_ref (… – …). Turbulence is … % of inflow resistance for methane at 1 million Sm³/d per well but only … % for hydrogen. Keeping methane D (Heinemann et al., 2025) would understate hydrogen inflow by about … % at high rates.
3. **Single wells:** at 220 bar, IA02 (s = 7) delivers … against … million Sm³/d for IA04 (s = −3.5): skin dominates differences between wells.
4. **Field sandface capacity:** across Heinemann's 180–275 bar cycle, injection at the BHP limits ranges from … to … million Sm³/d for hydrogen, against the … million Sm³/d average needed (notebook 02). Withdrawal …
5. **Interpretation:** if sandface capacity is well above both the historic methane rates and the hydrogen rates required, the **inflow is not the bottleneck**. The limit then sits in the tubing (next: gas-column effect) or the surface, consistent with the hypothesis. Capacity falls sharply only near the BHP envelope edges, which are reservoir rather than infrastructure constraints (A4).

**Limitations:** sandface only (no tubing, wellhead or surface); field-average k and h (A11); single tank (A9); pure gases (A1); pseudosteady state, no interference between wells; BHP envelope taken from methane operation (A4).

## Next

`feature/well-tubing`: bottomhole ↔ wellhead with an explicit gravity term. Combined with this inflow model it gives the **injection wellhead pressure required for hydrogen vs methane at the BHP limits**, the interim headline result, and validation V2 against Scafidi et al. (2023) Table 4.